In [0]:
main_path = ''

In [0]:
from pyspark.sql.functions import col, when, percentile, lower, count, avg, sum as sum_s, max as max_s
from pyspark.sql.types import IntegerType, DecimalType, FloatType
from pyspark.sql.window import Window

In [0]:
# Q1
# Implement: def add_valid_column(rides_df):
# Add a column:
# is_valid

# Rules:
# start_time < end_time  → True
# otherwise              → False

# Expected output:
# ride_id | start_time | end_time | is_valid

rides_df = spark.read.format('csv')\
                     .option('header','true')\
                     .load(main_path + '/rides.csv')

display(rides_df)

def add_valid_column(df):
    df = rides_df.withColumn('is_valid'
                                  ,when(col('end_time')>col('start_time')
                                       ,True
                                   )\
                                  .otherwise(False)     
                        )
    df = df.selectExpr('ride_id AS ride_id'
                      ,'start_time AS start_time'
                      ,'end_time AS  end_time'
                      ,'is_valid AS is_valid'
            )
    return df
display(add_valid_column(rides_df))

In [0]:
# temp_df = rides_df.withColumn('median', percentile(col('trip_cost'), 0.5).over(Window.partitionBy(col('ride_id'))))\
temp_df = rides_df.withColumn('trip_cost', col('trip_cost').cast(IntegerType()))
temp_ls = temp_df.approxQuantile('trip_cost', [0.25,0.5,0.75],0.05)

display(temp_ls)

q1 = temp_ls[0]
median = temp_ls[1]
q3 = temp_ls[2]
print(q1, median, q3)

In [0]:
# # Q2
# Implement:

# def remove_outliers(df):

# Use the trip_cost column.

# An observation is an outlier if:
# value < Q1 - 1.5 * IQR
# OR
# value > Q3 + 1.5 * IQR

display(rides_df)


def remove_outliers(df):
    df = df.withColumn('trip_cost', col('trip_cost').cast(IntegerType()) )
    temp_ls = df.approxQuantile('trip_cost', [0.25,0.5,0.75],0.05)
    display(temp_ls)
    q1 = temp_ls[0]
    median = temp_ls[1]
    q3 = temp_ls[2]
    iqr = q3 - q1
    print(q1, median, q3, iqr)

    df = rides_df.withColumn('outlier', 
                             when( (col('trip_cost') > (q3 + 1.5*iqr)) | 
                                   (col('trip_cost') < (q1 - 1.5*iqr))
                                  ,'Y'
                             )\
                            .otherwise('N')
                  
                  )
    return df

display(remove_outliers(rides_df))

In [0]:
# Q3 — Multiple Aggregations
# For each customer calculate:
# customer_id
# number_of_rides
# total_spend
# average_spend
# maximum_spend
# average_rating

# Only consider completed rides.

# Sort by:
# total_spend DESC
customers_df = spark.read.format('csv')\
                         .option('header','true')\
                         .load(main_path +'/customers.csv')
display(customers_df)

customer_ride_df = customers_df.join(rides_df
                                    ,rides_df.customer_id == customers_df.customer_id 
                                    ,'left'
                                )\
                               .drop(rides_df.customer_id)
display(customer_ride_df)

display(customer_ride_df.filter(lower(col('status'))=='completed').groupBy('customer_id').agg( count(col('ride_id')).alias('total_rides')
                                                                                              ,sum_s(col('trip_cost').cast(IntegerType())).alias('total_spend')
                                                                                              ,avg(col('trip_cost').cast(IntegerType())).alias('average_spend')
                                                                                              ,max_s(col('trip_cost').cast(IntegerType())).alias('max_spend')
                                                                                              ,avg(col('rating').cast(FloatType())).alias('average_rating')
                                                                                          )\
                                                                                         .orderBy(col('total_spend').desc())
)